In [ ]:
# 导入与实际来源校验；发现旧缓存时显式 Restart Kernel，不自动 reload。
from pathlib import Path
import sys
import platform
import hashlib
import numpy as np

core_dir = Path.cwd().resolve()
_loaded = {name: mod for name, mod in sys.modules.items() if name == 'fimrun' or name.startswith('fimrun.')}
_previous_identity = globals().get('_boris_import_identity')
if _loaded and _previous_identity is None:
    raise RuntimeError('fimrun was already imported without this identity check; Restart Kernel and run this cell first')
if _previous_identity is not None:
    for _name, _entry in _previous_identity['modules'].items():
        _mod = _loaded.get(_name)
        if _mod is None or id(_mod) != _entry['id'] or str(Path(_mod.__file__).resolve()) != _entry['file'] or hashlib.sha256(Path(_entry['file']).read_bytes()).hexdigest() != _entry['sha256']:
            raise RuntimeError('Imported module or source changed; Restart Kernel before continuing')
# 将选定源码放在首位；不替换已加载的模块。
sys.path[:] = [str(core_dir)] + [p for p in sys.path if Path(p).resolve() != core_dir]
from fimrun.micromagnetics import (
    Uniform, Antiparallel, Grid, FiM, FiMMaterial, Sublattice, FiMExchange, OnsiteCoupling,
    DMI, FiMAnisotropy, Demag, Zeeman, LegacyFixture,
    Simulation, RK4, MaxTransverseField, FiMSOT,
)
from fimrun.notebook import NotebookSession, summary, configuration_summary, module_identity
_boris_import_identity = module_identity(core_dir)
if _boris_import_identity['notebook_api'] != '1.1':
    raise RuntimeError('Notebook helper version mismatch; Restart Kernel with the intended project')
{'python': sys.executable, 'prefix': sys.prefix, 'numpy': np.__version__, 'numpy_source': np.__file__,
 'architecture': platform.machine(), 'actual_sources': _boris_import_identity}


In [ ]:
project_dir=Path.cwd()/'synthetic-contract-output'
task_name='synthetic-contract'
a=Sublattice(Ms=3e5,alpha=.01,gamma_H=2e5)
model=FiM(mesh=Grid(cells=(3,3,1),cell_size=(4e-9,4e-9,3e-9)),material=FiMMaterial(A=a,B=a),energy=())
initial=Antiparallel(Uniform((0.,0.,1.)))
preparation_dt=1e-15
preparation_output_settings=dict(scalars=(),every=None,snapshots=None,analysis=(),preview=True)


In [ ]:
# 创建新实验；重复执行会报错保护已有会话，不会悄悄清零。
if 'sim' in globals() and ('session' not in globals() or session.sim is not sim):
    raise RuntimeError('An independent sim already exists; keep it, or explicitly close and clear it before a new experiment')
session = NotebookSession.start(model=model, initial=initial, solver=RK4(dt=preparation_dt),
                                parent=project_dir / 'results', name=task_name,
                                output_settings=preparation_output_settings,
                                previous=globals().get('session'))
sim = session.sim
session.info()


In [ ]:
# 初态预览，零求场；关闭会话后须新建或恢复。
session.preview(sim)
